# DeepSeek-OCR VLLM 批量推理

使用 VLLM 进行高效的批 OCR 推理

## 1. 环境设置和导入

In [1]:
import os
import re
import json
import torch
from tqdm import tqdm
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
import glob

# 设置环境变量
os.environ['VLLM_USE_V1'] = '1'
# os.environ["CUDA_VISIBLE_DEVICES"] = '0,1,2,3'  # 使用 GPU 1

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current CUDA device:", torch.cuda.current_device())
print("Torch version:", torch.__version__)

CUDA available: True
CUDA device count: 4
Current CUDA device: 0
Torch version: 2.6.0+cu124


## 2. 添加 VLLM 模块路径

In [2]:
import sys
sys.path.append('/root/code/research/DeepSeek-OCR/DeepSeek-OCR-master/DeepSeek-OCR-vllm')

print("Python path:")
for p in sys.path[:5]:
    print(f"  {p}")

Python path:
  /usr/lib/python310.zip
  /usr/lib/python3.10
  /usr/lib/python3.10/lib-dynload
  
  /root/code/research/DeepSeek-OCR/.venv/lib/python3.10/site-packages


## 3. 导入 VLLM 和自定义模块

In [3]:
from vllm import LLM, SamplingParams
from vllm.model_executor.models.registry import ModelRegistry
from deepseek_ocr import DeepseekOCRForCausalLM
from process.image_process import DeepseekOCRProcessor
from process.ngram_norepeat import NoRepeatNGramLogitsProcessor

# 注册自定义模型
ModelRegistry.register_model("DeepseekOCRForCausalLM", DeepseekOCRForCausalLM)
print("✅ 模块导入成功")

INFO 11-17 05:54:07 [__init__.py:216] Automatically detected platform cuda.


ImportError: /root/code/research/DeepSeek-OCR/.venv/lib/python3.10/site-packages/vllm/_C.abi3.so: undefined symbol: _ZN3c106ivalue14ConstantString6createENSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE

## 4. 配置参数

In [ ]:
# 模型
MODEL_PATH = "../model/deepseek-ocr"

# 输入输出路径
INPUT_FOLDER = "../data/focus_benchmark_test/images_distorted"
OUTPUT_FOLDER = "../results/vllm_batch_results"

# 提示词
PROMPT = "<image>\\n<|grounding|>Convert the document to markdown. "

# 推理参数
CROP_MODE = False
MAX_CONCURRENCY = 8
NUM_WORKERS = 8

print(f"模型路径: {MODEL_PATH}")
print(f"输入文件夹: {INPUT_FOLDER}")
print(f"输出文件夹: {OUTPUT_FOLDER}")
print(f"批量大: {MAX_CONCURRENCY}")